## Collection of code I used for diagnosing problems with DESI-238 fits having NaN in them

In [ ]:
nan_chain_idx = jnp.unique(jnp.where(jnp.isnan(multi_chain_samples))[0])[0]
nan_chain = multi_chain_samples[nan_chain_idx]
nan_step1 = jnp.unique(jnp.where(jnp.isnan(nan_chain))[0])[0]

In [ ]:
nan_chain = multi_chain_samples[nan_chain_idx]
pre_post_nan_step = nan_chain[nan_step1-2:nan_step1+1]
# lp_mapped = jax.vmap(log_prob)
lp_and_grad_mapped = jax.vmap(jax.value_and_grad(log_prob))
print(pre_post_nan_step)
print(lp_and_grad_mapped(pre_post_nan_step))
nan_lp_point = nan_chain[nan_step1-1] #* Has finite positions but NaN log prob

In [ ]:
# import numpy as np
from jax import jit, lax
# from jax import numpy as jnp
# from jax import random
# from tensorflow_probability.substrates.jax import distributions as tfd, bijectors as tfb

# import gigalens.jax.simulator as sim
import gigalens.model
import functools
from typing import List, Dict
from lenstronomy.Util.kernel_util import subgrid_kernel
from objax.constants import ConvPadding
from objax.functional import average_pool_2d

class LensSimulatorTest(gigalens.simulator.LensSimulatorInterface):
    def __init__(
            self,
            phys_model: gigalens.model.PhysicalModel,
            sim_config: gigalens.simulator.SimulatorConfig,
            bs: int,
    ):
        super(LensSimulatorTest, self).__init__(phys_model, sim_config, bs)
        self.supersample = int(sim_config.supersample)
        self.transform_pix2angle = (
            jnp.eye(2) * sim_config.delta_pix
            if sim_config.transform_pix2angle is None
            else sim_config.transform_pix2angle
        )
        self.conversion_factor = jnp.linalg.det(self.transform_pix2angle)
        self.transform_pix2angle = self.transform_pix2angle / float(self.supersample)
        _, _, img_X, img_Y = self.get_coords(
            self.supersample, sim_config.num_pix, np.array(self.transform_pix2angle)
        )
        self.img_X = jnp.repeat(img_X[..., jnp.newaxis], bs, axis=-1)
        self.img_Y = jnp.repeat(img_Y[..., jnp.newaxis], bs, axis=-1)

        self.numPix = sim_config.num_pix
        self.bs = bs
        self.depth = sum([x.depth for x in self.phys_model.lens_light]) + sum(
            [x.depth for x in self.phys_model.source_light])
        self.kernel = None
        self.flat_kernel = None

        if sim_config.kernel is not None:
            kernel = subgrid_kernel(
                sim_config.kernel, sim_config.supersample, odd=True
            )[::-1, ::-1, jnp.newaxis, jnp.newaxis]
            self.kernel = jnp.repeat(kernel, self.depth, axis=2)
            self.flat_kernel = jnp.transpose(kernel, (2, 3, 0, 1))

    @functools.partial(jit, static_argnums=(0,))
    def _beta(self, lens_params: List[Dict]):
        beta_x, beta_y = self.img_X, self.img_Y
        for lens, p in zip(self.phys_model.lenses, lens_params):
            f_xi, f_yi = lens.deriv(self.img_X, self.img_Y, **p)
            beta_x, beta_y = beta_x - f_xi, beta_y - f_yi
        return beta_x, beta_y

    # @functools.partial(jit, static_argnums=(0,))
    def simulate(self, params, no_deflection=False):
        lens_params = params[0]
        lens_light_params, source_light_params = [], []
        if len(self.phys_model.lens_light) > 0:
            lens_light_params, source_light_params = params[1], params[2]
        else:
            source_light_params = params[1]
        beta_x, beta_y = self._beta(lens_params)
        if no_deflection:
            beta_x, beta_y = self.img_X, self.img_Y
        img = jnp.zeros_like(self.img_X)
        for lightModel, p in zip(self.phys_model.lens_light, lens_light_params):
            img += lightModel.light(self.img_X, self.img_Y, **p)
        for lightModel, p in zip(self.phys_model.source_light, source_light_params):
            img += lightModel.light(beta_x, beta_y, **p)
        img = jnp.transpose(img, (2, 0, 1))
        img = jnp.nan_to_num(img)
        ret = (
            lax.conv(img[:, jnp.newaxis, ...], self.flat_kernel, (1, 1), "SAME")
            if self.flat_kernel is not None
            else img
        )
        ret = (
            average_pool_2d(ret, size=self.supersample, padding=ConvPadding.SAME)
            if self.supersample != 1
            else ret
        )
        return jnp.squeeze(ret) * self.conversion_factor

    # @functools.partial(jit, static_argnums=(0,4,))
    def lstsq_simulate(
            self,
            params,
            observed_image,
            err_map,
            no_deflection=False,
    ):
        lens_params = params[0]
        lens_light_params, source_light_params = [], []
        if len(self.phys_model.lens_light) > 0:
            lens_light_params, source_light_params = params[1], params[2]
        else:
            source_light_params = params[1]
        beta_x, beta_y = self._beta(lens_params)
        print("beta_y:", jnp.all(jnp.isfinite(beta_y)))
        print("beta_x:", jnp.all(jnp.isfinite(beta_x)))

        
        if no_deflection:
            beta_x, beta_y = self.img_X, self.img_Y
        img = jnp.zeros((0, *self.img_X.shape))
        for lightModel, p in zip(self.phys_model.lens_light, lens_light_params):
            img = jnp.concatenate((img, lightModel.light(self.img_X, self.img_Y, **p)), axis=0)
        for lightModel, p in zip(self.phys_model.source_light, source_light_params):
            img = jnp.concatenate((img, lightModel.light(beta_x, beta_y, **p)), axis=0)

        print(img.shape)
        print("img:", jnp.all(jnp.isfinite(img), axis=(1,2)))
        # print("img:", img[4])
        img = jnp.nan_to_num(img) #! This is the problem. It 
        print("img post:", jnp.all(jnp.isfinite(img)))
        img = jnp.transpose(img, (3, 0, 1, 2))  # bs, n components, h, w
        
        ret = jax.lax.conv_general_dilated(img, self.kernel, (1, 1), padding='SAME', feature_group_count=self.depth,
                                           dimension_numbers=(
                                           'NCHW', 'HWOI', 'NCHW')) if self.flat_kernel is not None else img
        print("ret:", jnp.all(jnp.isfinite(ret)))
        ret = average_pool_2d(ret, size=(self.supersample, self.supersample),
                              padding="SAME") if self.supersample != 1 else ret
        ret = jnp.transpose(ret, (0, 2, 3, 1))  # bs, h, w, n components

        print("ret pooled:", jnp.all(jnp.isfinite(ret)))

        
        
        W = (1 / err_map)[..., jnp.newaxis]
        
        Y = jnp.reshape(observed_image * jnp.squeeze(W), (1, -1, 1))
        X = jnp.reshape((ret * W), (self.bs, -1, self.depth))
        
        Xt = jnp.transpose(X, (0, 2, 1))
        
        coeffs = (jnp.linalg.pinv(Xt @ X, rcond=1e-6) @ Xt @ Y)[..., 0]
        print("coeffs:", jnp.all(jnp.isfinite(coeffs)))
        

        ret = jnp.sum(ret * coeffs[:, jnp.newaxis, jnp.newaxis, :], axis=-1)
        print("ret final:", jnp.all(jnp.isfinite(ret)))
        
        return jnp.squeeze(ret), jnp.squeeze(coeffs)


class BackwardProbModelTest(gigalens.model.ProbabilisticModel):
    def __init__(
            self, prior: tfd.Distribution, observed_image, background_rms, exp_time
    ):
        super(BackwardProbModelTest, self).__init__(prior)
        err_map = jnp.sqrt(
            background_rms ** 2 + jnp.clip(observed_image, 0, np.inf) / exp_time
        )
        self.observed_dist = tfd.Independent(
            tfd.Normal(observed_image, err_map), reinterpreted_batch_ndims=2
        )
        self.observed_image = jnp.array(observed_image)
        self.err_map = jnp.array(err_map)
        example = prior.sample(seed=random.PRNGKey(0))
        self.pack_bij = tfb.pack_sequence_as(example)
        self.bij = tfb.Chain(
            [
                prior.experimental_default_event_space_bijector(),
                self.pack_bij,
            ]
        )

    # @functools.partial(jit, static_argnums=(0, 1))
    def log_prob(self, simulator: sim.LensSimulator, z):
        z = list(z.T)
        x = self.bij.forward(z)
        im_sim = simulator.lstsq_simulate(x, self.observed_image, self.err_map, no_deflection=True)[0] #* This is NaN
        log_like = self.observed_dist.log_prob(im_sim)
        log_prior = self.prior.log_prob(x) + self.bij.forward_log_det_jacobian(z)
        return log_like + log_prior, jnp.mean(
            ((im_sim - self.observed_image) / self.err_map) ** 2, axis=(-2, -1)
        )

lens_sim_test = LensSimulatorTest(phys_model, sim_config, bs=1)
prob_model_test = BackwardProbModelTest(prior, jnp.array(observed_img), background_rms=background_rms, exp_time=exp_time)
prob_model_test.log_prob(lens_sim_test, nan_lp_point)

In [ ]:
phys_model_for = PhysicalModel([epl.EPL(50), shear.Shear()], [sersic.SersicEllipse(),sersic.SersicEllipse(),sersic.SersicEllipse()], [sersic.SersicEllipse(), sersic.SersicEllipse()])
lens_sim_test_for = LensSimulatorTest(phys_model_for, sim_config, bs=1)

bad_sersic = prob_model_test.bij.forward(list(nan_lp_point[None,...].T))[2][1]
bad_sersic['Ie'] = jnp.array([1.0])
bad_sersic['n_sersic'] = jnp.array([0.3271/1.9992-0.001])

print("bn:", 1.9992 * bad_sersic['n_sersic'] - 0.3271)
print(bad_sersic)
# im = lens_sim_test.lstsq_simulate(prob_model_test.bij.forward(list(nan_lp_point.T)))
im = lens_sim_test_for.simulate([[], [], [bad_sersic]])
print("All finite: ", jnp.all(jnp.isfinite(im)))
plt.imshow(im)
plt.show()